# NALAPRO Project — Part 2
## Fine-tuning BERT-base for 20-Newsgroups Classification

---

## Tools used
| Tool | Purpose |
|------|---------|
| `transformers` (Hugging Face) | Pre-trained BERT, `Trainer`, tokeniser |
| `datasets` (Hugging Face) | `Dataset` / `DatasetDict` wrapper |
| `evaluate` (Hugging Face) | Accuracy & macro-F1 metrics |
| `scikit-learn` | Data split, confusion matrix |
| `PyTorch` | Underlying compute engine |
| `matplotlib` | Confusion matrix plot |
| `Claude (Anthropic)` | Scaffolding and code assistance |

**Reference:** *Hands-On Large Language Models* — Chapter 11  
→ https://github.com/HandsOnLLM/Hands-On-Large-Language-Models  
**In-class notebook:** `FineTuningClassification.ipynb`

---

## 📖 What is BERT and why does it beat Part 1?

BERT (Bidirectional Encoder Representations from Transformers, Devlin et al., 2018) is a
transformer encoder pre-trained on 3.3 billion tokens with two objectives:

| Objective | What BERT learns |
|-----------|------------------|
| **Masked Language Modelling (MLM)** | Predict randomly masked tokens from context on both sides |
| **Next Sentence Prediction (NSP)** | Determine if two sentences are consecutive |

The result is a model where **every word is represented in the context of its entire sentence**,
unlike Word2Vec (which only sees a fixed local window) or TF-IDF (which ignores position entirely).

### Three experiments: how much of BERT do we need to fine-tune?

| Run | What's trainable | Expected macro-F1 |
|-----|-----------------|-------------------|
| **A — Full fine-tune** | All 110M parameters + head | ~0.85 (best) |
| **B — Linear probe (frozen)** | Only 2-layer head (~15k params) | ~0.75 |
| **C — Partial freeze** | Top 2 encoder blocks + head (~7M params) | ~0.83 |

---


## Setup


In [ ]:
%pip install -q torch transformers datasets evaluate scikit-learn accelerate wandb


---
## Experiment tracking (in-notebook)

Rather than an external experiment-tracking server, all metrics are captured directly in this notebook.

**Per-run parameter breakdown (printed after each run completes)**

| Statistic | Description |
|-----------|-------------|
| Total params | All BERT parameters + classification head |
| Trainable | Parameters with `requires_grad=True` — updated by the optimiser |
| Frozen | Parameters with `requires_grad=False` — pre-trained values kept fixed |
| Head weights / biases | The `classifier.weight` (768×20) and `classifier.bias` (20,) layers |

**Visualisations produced after all three runs**
- Training curves: validation loss and macro-F1 per epoch for Runs A, B, C
- Classifier head weight distributions: histograms of the 15,360 head weights per run
- Classifier head bias bar chart: the 20 per-class bias values per run
- Trainable vs frozen parameter bar chart

In [ ]:
import os, random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_ID = "bert-base-uncased"   # 12 encoder layers, 110M parameters, uncased tokeniser
MAX_LEN  = 256                   # truncate to 256 WordPiece tokens (256 is a common sweet spot)


In [ ]:
# ── Google Drive setup (Google Colab only) ────────────────────────────────────────
# Mounts Drive so CSVs and plots persist across sessions and are visible to Part 3/4.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    NALAPRO_DIR = '/content/drive/MyDrive/NALAPRO'
    os.makedirs(NALAPRO_DIR, exist_ok=True)
    os.chdir(NALAPRO_DIR)
    _IN_COLAB = True
    CKPT_BASE = '/content'   # fast local storage for BERT checkpoints
    print(f'Google Colab detected.')
    print(f'Working directory (CSVs/plots) → {NALAPRO_DIR}')
    print(f'Checkpoint base (fast local)   → {CKPT_BASE}')
except ModuleNotFoundError:
    _IN_COLAB = False
    CKPT_BASE = os.getcwd()
    print(f'Not in Colab. Working directory / checkpoint base: {CKPT_BASE}')

---
## 1 · Data preparation

Reuse the same 20-Newsgroups data and the same train/val/test split as Part 1,
then wrap it in Hugging Face `Dataset` objects so the `Trainer` API can consume it.


In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

REMOVE = ("headers", "footers", "quotes")
ng_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SEED)
ng_test  = fetch_20newsgroups(subset="test",  remove=REMOVE, random_state=SEED)

class_names = ng_train.target_names
NUM_CLASSES = len(class_names)

# Same 90/10 stratified split used in Part 1 for consistency
X_tr, X_va, y_tr, y_va = train_test_split(
    ng_train.data, ng_train.target,
    test_size=0.10, stratify=ng_train.target, random_state=SEED,
)

# Wrap in HuggingFace Dataset objects.
# filter() removes any empty posts that result from the header/footer stripping.
raw = DatasetDict({
    "train":      Dataset.from_dict({"text": X_tr,          "label": y_tr.tolist()}),
    "validation": Dataset.from_dict({"text": X_va,          "label": y_va.tolist()}),
    "test":       Dataset.from_dict({"text": ng_test.data,   "label": ng_test.target.tolist()}),
}).filter(lambda ex: ex["text"] is not None and len(ex["text"].strip()) > 10)

print(raw)


---
## 2 · Tokenisation


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(batch):
    """
    Applied to batches of documents via Dataset.map().
    truncation=True: clip at max_length tokens (some posts are very long).
    We do NOT pad here — DataCollatorWithPadding does that per batch.
    """
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=["text"])

# The DataCollator pads each batch dynamically — shorter sequences don't waste computation
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)
print("\nExample — first 10 input_ids of the first training doc:")
print(tokenized["train"][0]["input_ids"][:10])
print("Decoded:", tokenizer.decode(tokenized["train"][0]["input_ids"][:10]))


---
## 3 · Evaluation metric

 **Macro-F1** as the primary metric because:
- It averages F1 across all 20 classes equally, regardless of class size.
- If the model completely ignores a small class, that gets heavily penalised.
- Accuracy alone can look good even if the model never predicts rare classes correctly.


In [ ]:
import evaluate as hf_evaluate

_acc = hf_evaluate.load("accuracy")
_f1  = hf_evaluate.load("f1")

def compute_metrics(eval_pred):
    """Called by the Trainer after each eval step."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  _acc.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1":  _f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


---
## Run A — Full fine-tuning

### TrainingArguments explained

| Argument | Value | Rationale |
|----------|-------|-----------|
| `learning_rate` | 2e-5 | Standard for BERT fine-tuning. Too large → catastrophic forgetting of pre-trained weights. |
| `num_train_epochs` | 3 | BERT converges in 2–4 epochs on text classification; more epochs overfit. |
| `weight_decay` | 0.01 | L2 regularisation — penalises large weights and helps generalisation. |
| `load_best_model_at_end` | True | After training, reload the checkpoint with the best val macro-F1 automatically. |
| `eval_strategy` | "epoch" | Evaluate on the validation set after every epoch. |


In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

model_A = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES
)

total_A = sum(p.numel() for p in model_A.parameters())
print(f'Total parameters: {total_A:,}  (all trainable in Run A)')

args_A = TrainingArguments(
    output_dir       = f'{CKPT_BASE}/ckpt_part2_full',
    learning_rate    = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    num_train_epochs  = 3,
    weight_decay      = 0.01,
    eval_strategy     = 'epoch',
    save_strategy     = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    greater_is_better      = True,
    logging_steps     = 50,
    fp16              = torch.cuda.is_available(),
    report_to         = 'none',
    seed              = SEED,
)

trainer_A = Trainer(
    model             = model_A,
    args              = args_A,
    train_dataset     = tokenized['train'],
    eval_dataset      = tokenized['validation'],
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
)

trainer_A.train()

In [ ]:
metrics_A_val  = trainer_A.evaluate(eval_dataset=tokenized['validation'])
metrics_A_test = trainer_A.evaluate(eval_dataset=tokenized['test'])
print('\n── Run A results ──────────────────────────────────')
print('Validation:', metrics_A_val)
print('Test:      ', metrics_A_test)
trainable_A = sum(p.numel() for p in model_A.parameters() if p.requires_grad)
print(f'  Total params: {total_A:,}  |  Trainable: {trainable_A:,}  (100.0%)')


---
## Run B — Frozen BERT (linear probe)

Here we **freeze every BERT parameter** and train only the small classification head.
This answers the question: *how good are BERT's pre-trained representations,
without any task-specific adaptation?*

In [ ]:
model_B = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES
)

# Freeze everything EXCEPT the classification head
for name, param in model_B.named_parameters():
    if name.startswith("classifier"):
        param.requires_grad = True   # train only the head
    else:
        param.requires_grad = False  # freeze BERT

trainable = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model_B.parameters() if not p.requires_grad)
print(f"Trainable parameters : {trainable:>12,}  ({100*trainable/(trainable+frozen):.3f}%)")
print(f"Frozen parameters    : {frozen:>12,}")
print("(The 'classifier.*' layers are the only trainable ones)")

# Print the trainable layer names
print("\nTrainable layers:")
for n, p in model_B.named_parameters():
    if p.requires_grad:
        print(f"  {n:40s}  shape={list(p.shape)}")


In [ ]:
args_B = TrainingArguments(
    output_dir = f'{CKPT_BASE}/ckpt_part2_frozen',
    learning_rate = 1e-3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    num_train_epochs  = 5,
    weight_decay      = 0.01,
    eval_strategy     = 'epoch',
    save_strategy     = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    greater_is_better      = True,
    logging_steps     = 50,
    fp16              = torch.cuda.is_available(),
    report_to         = 'none',
    seed              = SEED,
)

trainer_B = Trainer(
    model             = model_B,
    args              = args_B,
    train_dataset     = tokenized['train'],
    eval_dataset      = tokenized['validation'],
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
)

trainer_B.train()

In [ ]:
metrics_B_val  = trainer_B.evaluate(eval_dataset=tokenized['validation'])
metrics_B_test = trainer_B.evaluate(eval_dataset=tokenized['test'])
print('\n── Run B results ──────────────────────────────────')
print('Validation:', metrics_B_val)
print('Test:      ', metrics_B_test)
trainable_B = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
frozen_B    = sum(p.numel() for p in model_B.parameters() if not p.requires_grad)
print(f'  Trainable: {trainable_B:,}  |  Frozen: {frozen_B:,}  ({100*trainable_B/(trainable_B+frozen_B):.3f}% trainable)')


---
## Run C — Partial freeze (layers 0-9 frozen, top 2 encoder layers + head trainable)

This experiment sits *between* Run A (full fine-tune) and Run B (frozen BERT).
We freeze the first 10 encoder layers and allow the top 2 (layers 10–11) plus the
classification head to update.

Inspired by the class notebook `FineTuningClassification.ipynb` (L11).

In [ ]:
model_C = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES
)

FREEZE_CUTOFF = 165
for idx, (name, param) in enumerate(model_C.named_parameters()):
    param.requires_grad = (idx >= FREEZE_CUTOFF)

trainable_C = sum(p.numel() for p in model_C.parameters() if p.requires_grad)
frozen_C    = sum(p.numel() for p in model_C.parameters() if not p.requires_grad)
print(f'Run C — trainable: {trainable_C:,}  frozen: {frozen_C:,}')
print(f'Proportion trainable: {100*trainable_C/(trainable_C+frozen_C):.1f}%')

args_C = TrainingArguments(
    output_dir = f'{CKPT_BASE}/ckpt_part2_partial',
    learning_rate = 5e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    num_train_epochs  = 3,
    weight_decay      = 0.01,
    eval_strategy     = 'epoch',
    save_strategy     = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    greater_is_better      = True,
    logging_steps     = 50,
    fp16              = torch.cuda.is_available(),
    report_to         = 'none',
    seed              = SEED,
)

trainer_C = Trainer(
    model             = model_C,
    args              = args_C,
    train_dataset     = tokenized['train'],
    eval_dataset      = tokenized['validation'],
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
)

trainer_C.train()

In [ ]:
metrics_C_val  = trainer_C.evaluate(eval_dataset=tokenized['validation'])
metrics_C_test = trainer_C.evaluate(eval_dataset=tokenized['test'])
print('\n── Run C results ──────────────────────────────────')
print('Validation:', metrics_C_val)
print('Test:      ', metrics_C_test)
trainable_C = sum(p.numel() for p in model_C.parameters() if p.requires_grad)
frozen_C    = sum(p.numel() for p in model_C.parameters() if not p.requires_grad)
print(f'  Trainable: {trainable_C:,}  |  Frozen: {frozen_C:,}  ({100*trainable_C/(trainable_C+frozen_C):.1f}% trainable)')


---
## 4 · Comparison, confusion matrix & analysis


In [ ]:
import pandas as pd

# ── Parameter breakdown per run ───────────────────────────────────────────────────
def param_stats(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head_w    = model.classifier.weight.numel()
    head_b    = model.classifier.bias.numel()
    return total, trainable, total - trainable, head_w, head_b

tot_A, tr_A, fr_A, hw_A, hb_A = param_stats(model_A)
tot_B, tr_B, fr_B, hw_B, hb_B = param_stats(model_B)
tot_C, tr_C, fr_C, hw_C, hb_C = param_stats(model_C)

param_df = pd.DataFrame([
    {'Run': 'A — Full fine-tune',   'Strategy': 'All layers',          'Total':   f'{tot_A:,}', 'Trainable': f'{tr_A:,}', 'Frozen': f'{fr_A:,}', '% Train': '100.0%',                     'Head W': f'{hw_A:,}', 'Head b': f'{hb_A:,}', 'Val F1': f'{metrics_A_val["eval_macro_f1"]:.4f}', 'Test Acc': f'{metrics_A_test["eval_accuracy"]:.4f}', 'Test F1': f'{metrics_A_test["eval_macro_f1"]:.4f}'},
    {'Run': 'B — Linear probe',     'Strategy': 'Head only (frozen)',   'Total':   f'{tot_B:,}', 'Trainable': f'{tr_B:,}', 'Frozen': f'{fr_B:,}', '% Train': f'{100*tr_B/tot_B:.3f}%',     'Head W': f'{hw_B:,}', 'Head b': f'{hb_B:,}', 'Val F1': f'{metrics_B_val["eval_macro_f1"]:.4f}', 'Test Acc': f'{metrics_B_test["eval_accuracy"]:.4f}', 'Test F1': f'{metrics_B_test["eval_macro_f1"]:.4f}'},
    {'Run': 'C — Partial freeze',   'Strategy': 'Top 2 blocks + head', 'Total':   f'{tot_C:,}', 'Trainable': f'{tr_C:,}', 'Frozen': f'{fr_C:,}', '% Train': f'{100*tr_C/tot_C:.1f}%',     'Head W': f'{hw_C:,}', 'Head b': f'{hb_C:,}', 'Val F1': f'{metrics_C_val["eval_macro_f1"]:.4f}', 'Test Acc': f'{metrics_C_test["eval_accuracy"]:.4f}', 'Test F1': f'{metrics_C_test["eval_macro_f1"]:.4f}'},
])

print('─' * 115)
print('PARAMETER BREAKDOWN & PERFORMANCE — PART 2')
print('─' * 115)
print(param_df.to_string(index=False))

# ── Cross-part summary ────────────────────────────────────────────────────────────
try:
    p1 = pd.read_csv('part1_results.csv')
    best = p1.loc[p1['Macro-F1'].astype(float).idxmax()]
    p1_acc, p1_f1 = float(best['Test Acc']), float(best['Macro-F1'])
    p1_label = f"Part 1 — {best['Experiment']}"
except Exception:
    p1_acc, p1_f1, p1_label = 0.70, 0.69, 'Part 1 — Best (placeholder)'

summary = pd.DataFrame([
    {'Experiment': p1_label,                                  'Test Acc': f'{p1_acc:.4f}',                                     'Test Macro-F1': f'{p1_f1:.4f}'},
    {'Experiment': 'Part 2A — BERT full fine-tune',           'Test Acc': f'{metrics_A_test["eval_accuracy"]:.4f}',            'Test Macro-F1': f'{metrics_A_test["eval_macro_f1"]:.4f}'},
    {'Experiment': 'Part 2B — BERT linear probe (frozen)',    'Test Acc': f'{metrics_B_test["eval_accuracy"]:.4f}',            'Test Macro-F1': f'{metrics_B_test["eval_macro_f1"]:.4f}'},
    {'Experiment': 'Part 2C — BERT partial freeze',           'Test Acc': f'{metrics_C_test["eval_accuracy"]:.4f}',            'Test Macro-F1': f'{metrics_C_test["eval_macro_f1"]:.4f}'},
])
print('\n')
print(summary.to_string(index=False))
summary.to_csv('part2_results.csv', index=False)
print('\n→ Saved part2_results.csv')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Helper: extract per-epoch eval rows from trainer log history ──────────────────
def get_eval_log(trainer):
    return [r for r in trainer.state.log_history if 'eval_loss' in r]

eval_A = get_eval_log(trainer_A)
eval_B = get_eval_log(trainer_B)
eval_C = get_eval_log(trainer_C)

# ── Plot 1: Training curves — validation loss & macro-F1 per epoch ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
run_styles = [
    ('A — Full fine-tune', eval_A, '#e07b39', 'o'),
    ('B — Linear probe',   eval_B, '#2196F3', 's'),
    ('C — Partial freeze', eval_C, '#4CAF50', '^'),
]
for label, log, color, marker in run_styles:
    epochs = [r['epoch'] for r in log]
    axes[0].plot(epochs, [r['eval_loss'] for r in log],        marker=marker, color=color, label=label, lw=2)
    axes[1].plot(epochs, [r.get('eval_macro_f1', 0) for r in log], marker=marker, color=color, label=label, lw=2)

for ax, title, ylabel in [
    (axes[0], 'Validation Loss per Epoch',     'Cross-entropy loss'),
    (axes[1], 'Validation Macro-F1 per Epoch', 'Macro-F1'),
]:
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('BERT Fine-tuning Curves — Runs A, B, C', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part2_training_curves.png')

# ── Plot 2: Classifier head weight & bias distributions ──────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 13))
run_models = [
    ('Run A — Full fine-tune',   model_A, '#e07b39'),
    ('Run B — Linear probe',     model_B, '#2196F3'),
    ('Run C — Partial freeze',   model_C, '#4CAF50'),
]
for row_i, (label, model, color) in enumerate(run_models):
    head_w = model.classifier.weight.data.cpu().numpy().ravel()
    head_b = model.classifier.bias.data.cpu().numpy()
    ax_w, ax_b = axes[row_i, 0], axes[row_i, 1]

    ax_w.hist(head_w, bins=60, color=color, alpha=0.85, edgecolor='none')
    ax_w.axvline(head_w.mean(), color='k', ls='--', lw=1.5,
                 label=f'μ={head_w.mean():.4f}  σ={head_w.std():.4f}')
    ax_w.axvline(0, color='red', ls=':', lw=1, alpha=0.5)
    ax_w.set_title(f'{label} — Head Weights ({head_w.size:,} params)', fontweight='bold')
    ax_w.set_xlabel('Weight value'); ax_w.set_ylabel('Count')
    ax_w.legend(fontsize=9)

    ax_b.bar(range(len(head_b)), head_b, color=color, alpha=0.85, edgecolor='none')
    ax_b.axhline(0, color='red', ls=':', lw=1, alpha=0.5)
    ax_b.axhline(head_b.mean(), color='k', ls='--', lw=1.5,
                 label=f'μ={head_b.mean():.4f}  σ={head_b.std():.4f}')
    ax_b.set_title(f'{label} — Head Biases (20 params, one per class)', fontweight='bold')
    ax_b.set_xlabel('Class index (0–19)'); ax_b.set_ylabel('Bias value')
    ax_b.legend(fontsize=9)

plt.suptitle('Classifier Head Weights & Biases After Fine-tuning', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('part2_head_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part2_head_distributions.png')

# ── Plot 3: Trainable vs frozen parameter breakdown ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
run_labels = ['Run A\nFull fine-tune', 'Run B\nLinear probe', 'Run C\nPartial freeze']
tr_counts  = [tr_A, tr_B, tr_C]
fr_counts  = [fr_A, fr_B, fr_C]
x = np.arange(len(run_labels))
bars_t = ax.bar(x, tr_counts, 0.4, label='Trainable', color=['#e07b39', '#2196F3', '#4CAF50'], alpha=0.9)
ax.bar(x, fr_counts, 0.4, bottom=tr_counts, label='Frozen', color='#cccccc', alpha=0.6)
ax.bar_label(bars_t, labels=[f'{v/1e6:.1f}M' for v in tr_counts], padding=3, fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(run_labels, fontsize=11)
ax.set_ylabel('Parameter count')
ax.set_title('Trainable vs Frozen Parameters per Run', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('part2_param_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part2_param_breakdown.png')


In [ ]:
# ── Confusion matrix on Run A ─────────────────────────────────────────────────────

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred_output = trainer_A.predict(tokenized["test"])
y_pred = pred_output.predictions.argmax(axis=-1)
y_true = np.array(tokenized["test"]["label"])

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(13, 11))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False, values_format=".2f")
ax.set_title("BERT full fine-tune — normalised confusion matrix (test set)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("part2_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part2_confusion_matrix.png")


In [ ]:
# ── Per-class F1 scores ───────────────────────────────────────────────────────────
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
per_class = pd.DataFrame({
    "class": class_names,
    "f1_score": [report[c]["f1-score"] for c in class_names],
    "support":  [report[c]["support"]  for c in class_names],
}).sort_values("f1_score")

print(per_class.to_string(index=False))
print("\nHardest classes (lowest F1):", per_class["class"].head(5).tolist())
print("Easiest classes (highest F1):", per_class["class"].tail(5).tolist())


---
## ⚠️ Teardown — Run this before closing the notebook

> ### 🔴 STOP AND READ BEFORE YOU CLOSE
>
> **Google Colab charges per GPU-hour while the runtime is connected.**
> Even if you're not running cells, the runtime keeps billing.
>
> | Time forgotten | Approx cost (Colab Pro, T4 GPU) |
> |---|---|
> | 1 hour | ~$0.35 |
> | 1 day | ~$8 |
> | 1 week | ~$56 |
>
> **Run the two cells below, then disconnect your runtime.**

### What the teardown does
1. **Deletes model objects** from Python memory → frees GPU VRAM immediately
2. **Deletes checkpoint folders** from Colab disk → prevents disk-quota errors on next run
3. **Clears CUDA cache** → returns GPU memory to the OS
4. **Prints a checklist** → step-by-step confirmation you're fully shut down

### After running the teardown cells
Go to **Runtime → Disconnect and delete runtime** in the Colab menu bar.
That stops all billing for this session.



In [ ]:
# ── Step 1: Delete models & free GPU VRAM ────────────────────────────────────
import gc, torch

print('Freeing BERT models from GPU VRAM…')

for _varname in ['model_A', 'model_B', 'model_C',
                 'trainer_A', 'trainer_B', 'trainer_C']:
    if _varname in globals():
        obj = globals()[_varname]
        if hasattr(obj, 'model') and hasattr(obj.model, 'cpu'):
            obj.model.cpu()
        elif hasattr(obj, 'cpu'):
            obj.cpu()
        del globals()[_varname]
        print(f'  ✓ Deleted {_varname}')

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f'CUDA cache cleared. GPU memory still allocated: {allocated:.1f} MB')
else:
    print('No GPU detected. RAM freed via gc.collect().')

print('✓ Step 1 complete.')


In [ ]:
# ── Step 2: Delete checkpoint folders from Colab disk ────────────────────────
import shutil, os

ckpt_dirs = [
    f'{CKPT_BASE}/ckpt_part2_full',
    f'{CKPT_BASE}/ckpt_part2_frozen',
    f'{CKPT_BASE}/ckpt_part2_partial',
]

for d in ckpt_dirs:
    if os.path.isdir(d):
        size_mb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(d) for f in files
        ) / 1e6
        shutil.rmtree(d)
        print(f'✓ Deleted {d}  (freed ~{size_mb:.0f} MB)')
    else:
        print(f'  {d} not found (already deleted — OK)')

print()
print('=' * 65)
print('PART 2 TEARDOWN CHECKLIST')
print('=' * 65)
checks = [
    'BERT model_A, model_B, model_C removed from GPU VRAM',
    'Trainer objects deleted',
    'All three checkpoint folders deleted from Colab disk',
    'No Vertex AI endpoints (Part 2 uses Colab GPU only)',
]
for c in checks:
    print(f'  ✓ {c}')
print()
print('FINAL STEP (manual):')
print('  → Colab menu: Runtime → Disconnect and delete runtime')
print('  → Verify no unexpected spend: https://console.cloud.google.com/billing')
print()
print('Part 2 teardown complete.')